# 26. Volatility / Regime特徴量の比較
出典: FX (2).ipynb、セルindex [55]。保存出力は results/imported_20260909/ を参照。
研究履歴です。実行順・Notebook内変数・元の価格CSVに依存し、エラーが出たコードも保存しています。
自動判定の文言は元実験の判定であり、監査済みの結論ではありません。全セル一括実行は再現手順ではありません。
[USER_HOME] は匿名化した元のパスです。元Notebook内の案内や依頼文は研究資料として保持しています。


## 元セルindex 55


In [ ]:
# ============================================================
# FEATURE TOURNAMENT #2
# BASE vs +VOLATILITY vs +REGIME vs +VOLATILITY+REGIME
# 15m -> 30m / Nested Walk-Forward / 2026 Holdout
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression


# ============================================================
# SETTINGS
# ============================================================

SEED = 42

# 15分足 × 2本 = 30分後
HORIZON = 2

# 1トレードあたりコスト
BASE_COST_PCT = 0.004

THRESHOLDS = [
    0.55,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
]

SESSIONS = [
    "ALL",
    "UTC_13_24",
    "UTC_21_24",
    "EXCLUDE_08_13",
]

CALS = [
    "RAW",
    "PLATT",
    "ISOTONIC",
]

TEST_YEARS = list(range(2020, 2027))

DEV_YEARS = list(range(2020, 2026))

MIN_TRADES = 50


# ============================================================
# 1. bars を確認・正規化
# ============================================================

if "bars" not in globals():
    raise ValueError(
        "DataFrame 'bars' が見つかりません。"
        "前の15分足データ作成セルを先に実行してください。"
    )

if not isinstance(bars, pd.DataFrame):
    raise ValueError("'bars' がDataFrameではありません。")


B = bars.copy()

B.columns = [
    str(c).lower().strip()
    for c in B.columns
]


# timestamp列がある場合
if "timestamp" in B.columns:

    idx = pd.to_datetime(
        B["timestamp"],
        utc=True,
        errors="coerce"
    )

    good = idx.notna()

    B = B.loc[good].copy()

    B.index = idx[good]


# indexがtimestampの場合
else:

    B.index = pd.to_datetime(
        B.index,
        utc=True,
        errors="coerce"
    )

    B = B.loc[
        B.index.notna()
    ].copy()


# 列名の別名対応
rename_map = {
    "o": "open",
    "h": "high",
    "l": "low",
    "c": "close",
    "vol": "volume",
}

B = B.rename(
    columns={
        c: rename_map[c]
        for c in B.columns
        if c in rename_map
    }
)


required = [
    "open",
    "high",
    "low",
    "close",
]

missing = [
    c
    for c in required
    if c not in B.columns
]

if missing:

    raise ValueError(
        f"bars にOHLC列がありません: {missing}"
    )


for c in required:

    B[c] = pd.to_numeric(
        B[c],
        errors="coerce"
    )


if "volume" in B.columns:

    B["volume"] = pd.to_numeric(
        B["volume"],
        errors="coerce"
    )


B = B.dropna(
    subset=required
)

B = B[
    ~B.index.duplicated(
        keep="last"
    )
]

B = B.sort_index()


print("=" * 80)
print("DATA CHECK")
print("=" * 80)

print(
    f"Rows: {len(B):,}"
)

print(
    f"Period: {B.index.min()} -> {B.index.max()}"
)


# ============================================================
# 2. BASE FEATURE
# ============================================================

def calc_rsi(close, n):

    diff = close.diff()

    gain = (
        diff
        .clip(lower=0)
        .rolling(n)
        .mean()
    )

    loss = (
        -diff
        .clip(upper=0)
        .rolling(n)
        .mean()
    )

    rs = (
        gain /
        loss.replace(0, np.nan)
    )

    return (
        100 -
        100 / (1 + rs)
    ) / 100


def make_base_features(b):

    c = b["close"]
    o = b["open"]
    h = b["high"]
    l = b["low"]

    x = pd.DataFrame(
        index=b.index
    )

    # -------------------------
    # 過去リターン
    # -------------------------

    for k in [
        1,
        2,
        4,
        8,
        16,
    ]:

        x[f"ret_{k}"] = (
            c.pct_change(k)
        )


    # -------------------------
    # ローソク足形状
    # -------------------------

    candle_range = (
        h - l
    ).replace(
        0,
        np.nan
    )

    x["body"] = (
        (c - o) /
        o.replace(0, np.nan)
    )

    x["range"] = (
        (h - l) /
        c.replace(0, np.nan)
    )

    x["close_pos"] = (
        (c - l) /
        candle_range
    )


    # -------------------------
    # EMA
    # -------------------------

    for n in [
        5,
        10,
        20,
        48,
    ]:

        ema = (
            c
            .ewm(
                span=n,
                adjust=False,
                min_periods=n
            )
            .mean()
        )

        x[f"close_ema_{n}"] = (
            c / ema - 1
        )


    # -------------------------
    # RSI
    # -------------------------

    x["rsi_7"] = calc_rsi(
        c,
        7
    )

    x["rsi_14"] = calc_rsi(
        c,
        14
    )


    # -------------------------
    # Channel position
    # -------------------------

    for n in [
        8,
        16,
        32,
    ]:

        low_n = (
            l
            .rolling(n)
            .min()
        )

        high_n = (
            h
            .rolling(n)
            .max()
        )

        x[f"channel_pos_{n}"] = (
            (c - low_n) /
            (high_n - low_n)
            .replace(0, np.nan)
        )


    # -------------------------
    # 時間特徴
    # -------------------------

    hour = (
        b.index.hour +
        b.index.minute / 60
    )

    dow = (
        b.index.dayofweek
    )

    x["hour_sin"] = np.sin(
        2 * np.pi *
        hour / 24
    )

    x["hour_cos"] = np.cos(
        2 * np.pi *
        hour / 24
    )

    x["dow_sin"] = np.sin(
        2 * np.pi *
        dow / 7
    )

    x["dow_cos"] = np.cos(
        2 * np.pi *
        dow / 7
    )

    return x


# ------------------------------------------------------------
# 過去コードに明示的なChampion BASE featuresが残っていれば再利用
#
# genericな feature_cols は、
# 前回Trend特徴量が混ざっている可能性があるので意図的に使わない
# ------------------------------------------------------------

def try_existing_base():

    feature_list_names = [

        "HGB_BASE_FEATURES",

        "CHAMPION_FEATURES",

        "BASE_FEATURES",

        "HGB_BASE_FEATURE_COLS",

        "CHAMPION_FEATURE_COLS",

        "BASE_FEATURE_COLS",
    ]


    for list_name in feature_list_names:

        obj = globals().get(
            list_name
        )

        if not isinstance(
            obj,
            (
                list,
                tuple,
                pd.Index,
                np.ndarray,
            )
        ):

            continue


        cols = list(obj)


        if len(cols) < 5:

            continue


        for df_name in [

            "ml_data",

            "dataset",

            "data",

            "df",

            "feature_df",

            "X",
        ]:

            d = globals().get(
                df_name
            )

            if not isinstance(
                d,
                pd.DataFrame
            ):

                continue


            temp = d.copy()


            if "timestamp" in temp.columns:

                temp.index = pd.to_datetime(
                    temp["timestamp"],
                    utc=True,
                    errors="coerce"
                )

            else:

                try:

                    temp.index = pd.to_datetime(
                        temp.index,
                        utc=True,
                        errors="coerce"
                    )

                except Exception:

                    continue


            if all(
                c in temp.columns
                for c in cols
            ):

                print(
                    f"[BASE] 既存Champion featureを再利用"
                )

                print(
                    f"       Feature list: {list_name}"
                )

                print(
                    f"       DataFrame: {df_name}"
                )

                print(
                    f"       Features: {len(cols)}"
                )

                result = (
                    temp[cols]
                    .apply(
                        pd.to_numeric,
                        errors="coerce"
                    )
                    .reindex(B.index)
                )

                return result


    print(
        "[BASE] 明示的なChampion feature listが見つかりません。"
    )

    print(
        "       固定BASE特徴量を作成して比較します。"
    )

    return make_base_features(
        B
    )


BASE_X = try_existing_base()


print(
    f"BASE features: {BASE_X.shape[1]}"
)


# ============================================================
# 3. VOLATILITY FEATURES
# ============================================================

def make_volatility_features(b):

    c = b["close"]

    h = b["high"]

    l = b["low"]


    log_ret = (
        np.log(c)
        .diff()
    )


    x = pd.DataFrame(
        index=b.index
    )


    # Realized volatility
    for n in [
        4,
        8,
        16,
        32,
        64,
    ]:

        x[f"rv_{n}"] = (
            log_ret
            .rolling(n)
            .std()
        )


    # True Range
    previous_close = (
        c.shift(1)
    )


    true_range = pd.concat(

        [

            h - l,

            (
                h -
                previous_close
            ).abs(),

            (
                l -
                previous_close
            ).abs(),

        ],

        axis=1,

    ).max(
        axis=1
    )


    # ATR
    x["atr14_pct"] = (
        true_range
        .rolling(14)
        .mean()
        /
        c
    )

    x["atr28_pct"] = (
        true_range
        .rolling(28)
        .mean()
        /
        c
    )


    # Parkinson volatility
    parkinson = (

        np.log(
            h /
            l.replace(
                0,
                np.nan
            )
        ) ** 2

        /

        (
            4 *
            np.log(2)
        )
    )


    for n in [
        8,
        16,
        32,
    ]:

        x[f"parkinson_{n}"] = (

            np.sqrt(

                parkinson
                .rolling(n)
                .mean()

            )

        )


    # Short / Long volatility ratio
    rv4 = (
        log_ret
        .rolling(4)
        .std()
    )

    rv8 = (
        log_ret
        .rolling(8)
        .std()
    )

    rv32 = (
        log_ret
        .rolling(32)
        .std()
    )


    x["vol_ratio_4_32"] = (

        rv4 /

        rv32.replace(
            0,
            np.nan
        )

    )


    x["vol_ratio_8_32"] = (

        rv8 /

        rv32.replace(
            0,
            np.nan
        )

    )


    # Range expansion
    range_pct = (
        (h - l) /
        c
    )


    x["range_z20"] = (

        range_pct -

        range_pct
        .rolling(20)
        .mean()

    ) / (

        range_pct
        .rolling(20)
        .std()
        .replace(
            0,
            np.nan
        )

    )


    return x


VOL_X = make_volatility_features(
    B
)


print(
    f"Volatility features: {VOL_X.shape[1]}"
)


# ============================================================
# 4. REGIME FEATURES
# ============================================================

def make_regime_features(b):

    c = b["close"]

    h = b["high"]

    l = b["low"]


    ret = (
        c.pct_change()
    )


    x = pd.DataFrame(
        index=b.index
    )


    # -------------------------
    # Trend strength
    # -------------------------

    ema12 = (
        c
        .ewm(
            span=12,
            adjust=False,
            min_periods=12
        )
        .mean()
    )

    ema48 = (
        c
        .ewm(
            span=48,
            adjust=False,
            min_periods=48
        )
        .mean()
    )


    previous_close = (
        c.shift(1)
    )


    true_range = pd.concat(

        [

            h - l,

            (
                h -
                previous_close
            ).abs(),

            (
                l -
                previous_close
            ).abs(),

        ],

        axis=1,

    ).max(
        axis=1
    )


    atr14 = (
        true_range
        .rolling(14)
        .mean()
    )


    trend_gap = (
        ema12 -
        ema48
    )


    trend_strength = (

        trend_gap.abs()

        /

        atr14.replace(
            0,
            np.nan
        )

    )


    x["reg_trend_dir"] = (
        np.sign(
            trend_gap
        )
    )

    x["reg_trend_strength"] = (
        trend_strength
    )


    # -------------------------
    # Kaufman efficiency ratio
    # -------------------------

    for n in [
        16,
        32,
    ]:

        directional = (

            c -

            c.shift(n)

        ).abs()


        path = (

            c
            .diff()
            .abs()
            .rolling(n)
            .sum()

        )


        x[f"reg_efficiency_{n}"] = (

            directional

            /

            path.replace(
                0,
                np.nan
            )

        )


    # -------------------------
    # Volatility regime
    # -------------------------

    rv16 = (

        np.log(c)
        .diff()
        .rolling(16)
        .std()

    )


    # 過去だけで閾値決定
    vol_q33 = (

        rv16
        .rolling(
            192,
            min_periods=64
        )
        .quantile(0.33)
        .shift(1)

    )


    vol_q67 = (

        rv16
        .rolling(
            192,
            min_periods=64
        )
        .quantile(0.67)
        .shift(1)

    )


    trend_q33 = (

        trend_strength
        .rolling(
            192,
            min_periods=64
        )
        .quantile(0.33)
        .shift(1)

    )


    trend_q67 = (

        trend_strength
        .rolling(
            192,
            min_periods=64
        )
        .quantile(0.67)
        .shift(1)

    )


    x["reg_low_vol"] = (
        rv16 <
        vol_q33
    ).astype(float)


    x["reg_high_vol"] = (
        rv16 >
        vol_q67
    ).astype(float)


    x["reg_ranging"] = (
        trend_strength <
        trend_q33
    ).astype(float)


    x["reg_trending"] = (
        trend_strength >
        trend_q67
    ).astype(float)


    # -------------------------
    # Direction persistence
    # -------------------------

    x["reg_sign_balance_16"] = (

        np.sign(ret)
        .rolling(16)
        .mean()

    )


    # Interaction
    x["reg_trend_x_highvol"] = (

        x["reg_trend_dir"]

        *

        x["reg_high_vol"]

    )


    x["reg_trend_x_lowvol"] = (

        x["reg_trend_dir"]

        *

        x["reg_low_vol"]

    )


    return x


REGIME_X = make_regime_features(
    B
)


print(
    f"Regime features: {REGIME_X.shape[1]}"
)


# ============================================================
# 5. FEATURE SETS
# ============================================================

FEATURE_SETS = {

    "BASE":

        BASE_X,


    "BASE_PLUS_VOL":

        pd.concat(
            [
                BASE_X,
                VOL_X
            ],
            axis=1
        ),


    "BASE_PLUS_REGIME":

        pd.concat(
            [
                BASE_X,
                REGIME_X
            ],
            axis=1
        ),


    "BASE_PLUS_VOL_REGIME":

        pd.concat(
            [
                BASE_X,
                VOL_X,
                REGIME_X
            ],
            axis=1
        ),
}


# ============================================================
# 6. TARGET
# ============================================================

FWD_RETURN = (

    B["close"]
    .shift(-HORIZON)

    /

    B["close"]

    - 1

)


TARGET = (

    FWD_RETURN >
    0

).astype(int)


# ============================================================
# 7. HGB MODEL
# ============================================================

def get_model_template():

    candidates = []


    for name, obj in list(
        globals().items()
    ):

        if isinstance(
            obj,
            HistGradientBoostingClassifier
        ):

            score = 0

            lower_name = (
                name.lower()
            )

            if (
                "hgb" in lower_name
                or
                "hist" in lower_name
            ):

                score += 10


            if (
                "champ" in lower_name
                or
                "best" in lower_name
            ):

                score += 5


            candidates.append(
                (
                    score,
                    name,
                    obj
                )
            )


    if len(candidates) > 0:

        candidates.sort(
            key=lambda x: x[0],
            reverse=True
        )

        _, name, obj = (
            candidates[0]
        )

        print(
            f"[MODEL] HGB parameters reused from '{name}'"
        )

        return clone(obj)


    print(
        "[MODEL] Existing HGB not found."
    )

    print(
        "[MODEL] Using fallback HGB."
    )


    return HistGradientBoostingClassifier(

        learning_rate=0.05,

        max_iter=220,

        max_leaf_nodes=15,

        min_samples_leaf=80,

        l2_regularization=1.0,

        early_stopping=True,

        random_state=SEED,
    )


MODEL_TEMPLATE = (
    get_model_template()
)


# ============================================================
# 8. CALIBRATION
# ============================================================

def apply_calibration(
    method,
    p_fit,
    y_fit,
    p_apply
):

    p_fit = np.asarray(
        p_fit
    )

    p_apply = np.asarray(
        p_apply
    )

    y_fit = np.asarray(
        y_fit
    )


    if method == "RAW":

        return p_apply


    if len(
        np.unique(
            y_fit
        )
    ) < 2:

        return p_apply


    if method == "PLATT":

        model = (
            LogisticRegression()
        )

        model.fit(
            p_fit.reshape(
                -1,
                1
            ),
            y_fit
        )

        return (

            model
            .predict_proba(

                p_apply.reshape(
                    -1,
                    1
                )

            )[:, 1]

        )


    if method == "ISOTONIC":

        if len(
            np.unique(
                p_fit
            )
        ) < 10:

            return p_apply


        model = (
            IsotonicRegression(
                out_of_bounds="clip"
            )
        )

        model.fit(
            p_fit,
            y_fit
        )

        return (
            model.predict(
                p_apply
            )
        )


    return p_apply


# ============================================================
# 9. SESSION FILTER
# ============================================================

def session_mask(
    index,
    session
):

    hour = (
        index.hour
    )


    if session == "ALL":

        return np.ones(
            len(index),
            dtype=bool
        )


    if session == "UTC_13_24":

        return (
            hour >= 13
        )


    if session == "UTC_21_24":

        return (
            hour >= 21
        )


    if session == "EXCLUDE_08_13":

        return ~(
            (
                hour >= 8
            )
            &
            (
                hour < 13
            )
        )


    return np.ones(
        len(index),
        dtype=bool
    )


# ============================================================
# 10. METRICS
# ============================================================

def calc_metrics(
    returns
):

    r = pd.Series(
        returns,
        dtype=float
    )

    r = (
        r
        .replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )
        .dropna()
    )


    if len(r) == 0:

        return {

            "trades": 0,

            "win_rate": np.nan,

            "avg_return": np.nan,

            "profit_factor": np.nan,

            "growth": np.nan,

            "max_dd": np.nan,

            "return_to_dd": np.nan,
        }


    positive_sum = (
        r[
            r > 0
        ].sum()
    )

    negative_sum = (

        -r[
            r < 0
        ].sum()

    )


    if negative_sum > 0:

        pf = (
            positive_sum /
            negative_sum
        )

    else:

        pf = np.inf


    equity = (

        1 + r

    ).cumprod()


    drawdown = (

        equity

        /

        equity.cummax()

        - 1

    )


    growth = (

        equity.iloc[-1]

        - 1

    )


    max_dd = (
        drawdown.min()
    )


    if max_dd < 0:

        return_to_dd = (

            growth /

            abs(max_dd)

        )

    else:

        return_to_dd = np.inf


    return {

        "trades":

            len(r),

        "win_rate":

            (
                r > 0
            ).mean(),

        "avg_return":

            r.mean(),

        "profit_factor":

            pf,

        "growth":

            growth,

        "max_dd":

            max_dd,

        "return_to_dd":

            return_to_dd,
    }


# ============================================================
# 11. TRADE GENERATION
# ============================================================

def make_trades(
    index,
    probability,
    future_return,
    threshold,
    session,
    cost_pct
):

    p = np.asarray(
        probability
    )

    future_return = np.asarray(
        future_return
    )


    side = np.zeros(
        len(p),
        dtype=int
    )


    side[
        p >= threshold
    ] = 1


    side[
        p <= 1 - threshold
    ] = -1


    take = (

        (side != 0)

        &

        session_mask(
            index,
            session
        )

        &

        np.isfinite(p)

        &

        np.isfinite(
            future_return
        )

    )


    result = pd.DataFrame(
        index=index[
            take
        ]
    )


    result["gross_return"] = (

        side[take]

        *

        future_return[take]

    )


    result["net_return"] = (

        result[
            "gross_return"
        ]

        -

        cost_pct / 100

    )


    return result


# ============================================================
# 12. VALIDATION SCORE
# ============================================================

def validation_score(
    m
):

    if (
        m["trades"]
        <
        MIN_TRADES
    ):

        return -np.inf


    pf = (
        m[
            "profit_factor"
        ]
    )


    if not np.isfinite(pf):

        pf = 10


    pf = min(
        pf,
        10
    )


    return_dd = (
        m[
            "return_to_dd"
        ]
    )


    if not np.isfinite(
        return_dd
    ):

        return_dd = 20


    return_dd = min(
        max(
            return_dd,
            0
        ),
        20
    )


    average = (
        m[
            "avg_return"
        ]
    )


    score = (

        0.45
        *
        np.log(
            max(
                pf,
                1e-6
            )
        )

        +

        0.35
        *
        np.log1p(
            return_dd
        )

        +

        0.20
        *
        np.tanh(
            average
            *
            10000
        )

    )


    return score


# ============================================================
# 13. WALK-FORWARD
# ============================================================

def run_feature_set(
    feature_set_name,
    X
):

    D = (

        X.loc[
            :,
            ~X.columns.duplicated()
        ]
        .replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )
        .copy()
    )


    D["__target"] = TARGET

    D["__future_return"] = FWD_RETURN

    D["__year"] = (
        D.index.year
    )


    D = D[
        D[
            "__future_return"
        ].notna()
    ].copy()


    feature_cols = [

        c

        for c
        in X.columns

        if (
            c in D.columns
            and
            D[c].notna().any()
        )

    ]


    annual_results = []

    all_trades = []


    print()

    print(
        "=" * 80
    )

    print(
        feature_set_name
    )

    print(
        "=" * 80
    )


    for test_year in TEST_YEARS:


        train = D[

            (
                D["__year"]
                >=
                2016
            )

            &

            (
                D["__year"]
                <=
                test_year - 2
            )

        ]


        validation = D[
            D["__year"]
            ==
            test_year - 1
        ].sort_index()


        test = D[
            D["__year"]
            ==
            test_year
        ].sort_index()


        if (
            len(train) < 5000
            or
            len(validation) < 500
            or
            len(test) < 500
        ):

            print(
                f"[SKIP] {test_year}: data不足"
            )

            continue


        # ----------------------------------------------------
        # validation yearを
        # calibration 60%
        # parameter selection 40%
        # に分割
        # ----------------------------------------------------

        cut = int(
            len(validation)
            *
            0.60
        )


        val_cal = (
            validation.iloc[
                :cut
            ]
        )


        val_select = (
            validation.iloc[
                cut:
            ]
        )


        model = clone(
            MODEL_TEMPLATE
        )


        model.fit(

            train[
                feature_cols
            ],

            train[
                "__target"
            ].astype(int)

        )


        p_cal_raw = (

            model
            .predict_proba(

                val_cal[
                    feature_cols
                ]

            )[:, 1]

        )


        p_select_raw = (

            model
            .predict_proba(

                val_select[
                    feature_cols
                ]

            )[:, 1]

        )


        p_test_raw = (

            model
            .predict_proba(

                test[
                    feature_cols
                ]

            )[:, 1]

        )


        try:

            auc = roc_auc_score(

                test[
                    "__target"
                ],

                p_test_raw

            )

        except Exception:

            auc = np.nan


        best = None


        # ----------------------------------------------------
        # Calibration
        # Threshold
        # Session
        # をvalidationだけで選択
        # ----------------------------------------------------

        for calibration in CALS:


            p_select = apply_calibration(

                calibration,

                p_cal_raw,

                val_cal[
                    "__target"
                ].values,

                p_select_raw

            )


            p_test = apply_calibration(

                calibration,

                p_cal_raw,

                val_cal[
                    "__target"
                ].values,

                p_test_raw

            )


            for threshold in THRESHOLDS:


                for session in SESSIONS:


                    validation_trades = make_trades(

                        val_select.index,

                        p_select,

                        val_select[
                            "__future_return"
                        ].values,

                        threshold,

                        session,

                        BASE_COST_PCT

                    )


                    m = calc_metrics(

                        validation_trades[
                            "net_return"
                        ]

                    )


                    score = validation_score(
                        m
                    )


                    if (
                        best is None
                        or
                        score
                        >
                        best["score"]
                    ):

                        best = {

                            "score":
                                score,

                            "calibration":
                                calibration,

                            "threshold":
                                threshold,

                            "session":
                                session,

                            "p_test":
                                p_test,
                        }


        # ----------------------------------------------------
        # Completely untouched TEST YEAR
        # ----------------------------------------------------

        test_trades = make_trades(

            test.index,

            best[
                "p_test"
            ],

            test[
                "__future_return"
            ].values,

            best[
                "threshold"
            ],

            best[
                "session"
            ],

            BASE_COST_PCT

        )


        test_metrics = calc_metrics(

            test_trades[
                "net_return"
            ]

        )


        test_trades[
            "feature_set"
        ] = feature_set_name


        test_trades[
            "test_year"
        ] = test_year


        all_trades.append(
            test_trades
        )


        annual_results.append({

            "feature_set":
                feature_set_name,

            "test_year":
                test_year,

            "validation_year":
                test_year - 1,

            "calibration":
                best[
                    "calibration"
                ],

            "threshold":
                best[
                    "threshold"
                ],

            "session":
                best[
                    "session"
                ],

            "auc":
                auc,

            **test_metrics

        })


        print(
            f"{test_year} | "
            f"{best['calibration']:<8} "
            f"Threshold={best['threshold']:.2f} "
            f"{best['session']:<13} "
            f"AUC={auc:.4f} "
            f"Trades={test_metrics['trades']:4d} "
            f"PF={test_metrics['profit_factor']:.3f} "
            f"Avg={test_metrics['avg_return']*100:.5f}% "
            f"DD={test_metrics['max_dd']*100:.3f}%"
        )


    annual_results = pd.DataFrame(
        annual_results
    )


    all_trades = pd.concat(
        all_trades
    ).sort_index()


    return (
        annual_results,
        all_trades
    )


# ============================================================
# 14. RUN ALL FEATURE SETS
# ============================================================

ANNUAL_LIST = []

TRADE_LIST = []


for name, X in FEATURE_SETS.items():

    annual_result, trade_result = (
        run_feature_set(
            name,
            X
        )
    )

    ANNUAL_LIST.append(
        annual_result
    )

    TRADE_LIST.append(
        trade_result
    )


ANNUAL = pd.concat(
    ANNUAL_LIST,
    ignore_index=True
)


TRADES = pd.concat(
    TRADE_LIST
).sort_index()


# ============================================================
# 15. DEVELOPMENT SUMMARY
# ============================================================

summary_rows = []


for name in FEATURE_SETS.keys():


    annual_part = ANNUAL[

        (
            ANNUAL[
                "feature_set"
            ]
            ==
            name
        )

        &

        ANNUAL[
            "test_year"
        ].isin(
            DEV_YEARS
        )

    ]


    trade_part = TRADES[

        (
            TRADES[
                "feature_set"
            ]
            ==
            name
        )

        &

        TRADES[
            "test_year"
        ].isin(
            DEV_YEARS
        )

    ]


    combined_metrics = calc_metrics(

        trade_part[
            "net_return"
        ]

    )


    summary_rows.append({

        "feature_set":
            name,

        "years":
            len(
                annual_part
            ),

        "mean_auc":
            annual_part[
                "auc"
            ].mean(),

        "positive_years":

            int(
                (
                    annual_part[
                        "avg_return"
                    ]
                    >
                    0
                ).sum()
            ),

        "pf_above_1_years":

            int(
                (
                    annual_part[
                        "profit_factor"
                    ]
                    >
                    1
                ).sum()
            ),

        **combined_metrics

    })


SUMMARY = pd.DataFrame(
    summary_rows
)


print()

print(
    "=" * 80
)

print(
    "DEVELOPMENT SUMMARY 2020-2025"
)

print(
    "=" * 80
)

print(
    SUMMARY.to_string(
        index=False
    )
)


# ============================================================
# 16. 2026 HOLDOUT
# ============================================================

CONFIRMATION = ANNUAL[

    ANNUAL[
        "test_year"
    ]
    ==
    2026

].copy()


print()

print(
    "=" * 80
)

print(
    "FINAL CONFIRMATION 2026"
)

print(
    "=" * 80
)

print(
    CONFIRMATION.to_string(
        index=False
    )
)


# ============================================================
# 17. COST STRESS
# ============================================================

cost_rows = []


for name in FEATURE_SETS.keys():


    trade_part = TRADES[

        (
            TRADES[
                "feature_set"
            ]
            ==
            name
        )

        &

        TRADES[
            "test_year"
        ].isin(
            DEV_YEARS
        )

    ]


    for cost_x in [
        1.0,
        1.5,
        2.0,
    ]:


        stressed_return = (

            trade_part[
                "gross_return"
            ]

            -

            (
                BASE_COST_PCT
                *
                cost_x
            )

            /
            100

        )


        m = calc_metrics(
            stressed_return
        )


        cost_rows.append({

            "feature_set":
                name,

            "cost_x":
                cost_x,

            "cost_pct":

                BASE_COST_PCT
                *
                cost_x,

            **m

        })


COST = pd.DataFrame(
    cost_rows
)


print()

print(
    "=" * 80
)

print(
    "COST STRESS"
)

print(
    "=" * 80
)

print(
    COST.to_string(
        index=False
    )
)


# ============================================================
# 18. AUTOMATIC FEATURE DECISION
# ============================================================

BASE_DEV = (

    SUMMARY[
        SUMMARY[
            "feature_set"
        ]
        ==
        "BASE"
    ]
    .iloc[0]

)


BASE_2026 = (

    CONFIRMATION[
        CONFIRMATION[
            "feature_set"
        ]
        ==
        "BASE"
    ]
    .iloc[0]

)


decision_rows = []


print()

print(
    "=" * 80
)

print(
    "AUTOMATIC FEATURE DECISION"
)

print(
    "=" * 80
)


for name in [

    "BASE_PLUS_VOL",

    "BASE_PLUS_REGIME",

    "BASE_PLUS_VOL_REGIME",

]:


    development = (

        SUMMARY[
            SUMMARY[
                "feature_set"
            ]
            ==
            name
        ]
        .iloc[0]

    )


    confirmation = (

        CONFIRMATION[
            CONFIRMATION[
                "feature_set"
            ]
            ==
            name
        ]
        .iloc[0]

    )


    development_checks = {

        "AUC":

            development[
                "mean_auc"
            ]

            >

            BASE_DEV[
                "mean_auc"
            ],


        "PF":

            development[
                "profit_factor"
            ]

            >

            BASE_DEV[
                "profit_factor"
            ],


        "AVG_RETURN":

            development[
                "avg_return"
            ]

            >

            BASE_DEV[
                "avg_return"
            ],


        "RETURN_DD":

            development[
                "return_to_dd"
            ]

            >

            BASE_DEV[
                "return_to_dd"
            ],


        "POSITIVE_YEARS":

            development[
                "positive_years"
            ]

            >=

            BASE_DEV[
                "positive_years"
            ],
    }


    confirmation_checks = {

        "AUC":

            confirmation[
                "auc"
            ]

            >

            BASE_2026[
                "auc"
            ],


        "PF":

            confirmation[
                "profit_factor"
            ]

            >

            BASE_2026[
                "profit_factor"
            ],


        "AVG_RETURN":

            confirmation[
                "avg_return"
            ]

            >

            BASE_2026[
                "avg_return"
            ],


        "RETURN_DD":

            confirmation[
                "return_to_dd"
            ]

            >

            BASE_2026[
                "return_to_dd"
            ],
    }


    development_wins = (
        sum(
            development_checks.values()
        )
    )


    confirmation_wins = (
        sum(
            confirmation_checks.values()
        )
    )


    pf_2x = COST[

        (
            COST[
                "feature_set"
            ]
            ==
            name
        )

        &

        (
            COST[
                "cost_x"
            ]
            ==
            2.0
        )

    ][
        "profit_factor"
    ].iloc[0]


    passed = (

        development_wins
        >=
        3

        and

        confirmation_wins
        >=
        3

        and

        pf_2x
        >
        1

    )


    decision_rows.append({

        "feature_set":
            name,

        "development_wins":
            development_wins,

        "holdout_wins":
            confirmation_wins,

        "pf_2x_cost":
            pf_2x,

        "passed":
            passed,

    })


    print()

    print(
        name
    )

    print(
        "Development:",
        development_checks,
        f"=> {development_wins}/5"
    )

    print(
        "2026 Holdout:",
        confirmation_checks,
        f"=> {confirmation_wins}/4"
    )

    print(
        f"2x Cost PF: {pf_2x:.3f}"
    )

    print(
        "PASS:",
        passed
    )


DECISION = pd.DataFrame(
    decision_rows
)


passed = DECISION[
    DECISION[
        "passed"
    ]
]


print()

print(
    "-" * 80
)


if len(
    passed
) == 0:

    print(
        "RESULT:"
    )

    print(
        "VOLATILITY / REGIME VALUE IS NOT YET CLEAR"
    )

    print(
        "BASEを維持します。"
    )

    print(
        "次は Trend × Regime interaction を別検証します。"
    )


else:

    candidate = passed.merge(

        SUMMARY[[
            "feature_set",
            "return_to_dd",
            "profit_factor",
            "mean_auc",
        ]],

        on="feature_set"

    )


    candidate = candidate.sort_values(

        [
            "return_to_dd",
            "profit_factor",
            "mean_auc",
        ],

        ascending=False

    )


    winner = (
        candidate.iloc[0][
            "feature_set"
        ]
    )


    print(
        "RESULT:"
    )

    print(
        f"{winner} HAS CLEAR INCREMENTAL OOS VALUE"
    )

    print(
        "次はこのwinnerをChampion Pipelineへ戻して正式確認します。"
    )


# ============================================================
# 19. Notebook保存
# ============================================================

FEATURE_TOURNAMENT_2_ANNUAL = (
    ANNUAL.copy()
)

FEATURE_TOURNAMENT_2_SUMMARY = (
    SUMMARY.copy()
)

FEATURE_TOURNAMENT_2_CONFIRMATION = (
    CONFIRMATION.copy()
)

FEATURE_TOURNAMENT_2_COST = (
    COST.copy()
)

FEATURE_TOURNAMENT_2_DECISION = (
    DECISION.copy()
)

FEATURE_TOURNAMENT_2_TRADES = (
    TRADES.copy()
)


print()

print(
    "=" * 80
)

print(
    "検証終了"
)

print(
    "=" * 80
)

print(
    "保存されたNotebook変数:"
)

print(
    "FEATURE_TOURNAMENT_2_ANNUAL"
)

print(
    "FEATURE_TOURNAMENT_2_SUMMARY"
)

print(
    "FEATURE_TOURNAMENT_2_CONFIRMATION"
)

print(
    "FEATURE_TOURNAMENT_2_COST"
)

print(
    "FEATURE_TOURNAMENT_2_DECISION"
)

print(
    "FEATURE_TOURNAMENT_2_TRADES"
)
